# 02 · Chạy thí nghiệm (Bảng 11)

Thứ tự khuyến nghị: (1) chạy thử trên dữ liệu giả lập để chắc chắn môi trường ổn,
(2) đo thời gian trên 1 cửa sổ dữ liệu thật, (3) chạy baseline, LightGBM, LSTM, (4) độ bền.
Kết quả được ghi thẳng vào Google Drive nên không mất khi Colab ngắt kết nối;
chạy lại lệnh sẽ ghi đè tệp của đúng cấu hình đó.

Lưu ý: Colab miễn phí có thể ngắt sau vài giờ. Chạy theo từng nhóm ô, mỗi ô một cấu hình.

In [ ]:
# ===== THIẾT LẬP CHUNG (chạy đầu tiên mỗi khi mở notebook) =====
GITHUB_USER = "<USERNAME>"          # <-- đổi thành tên tài khoản GitHub của bạn
REPO = "m5-calendar-forecasting"
DRIVE_DIR = "/content/drive/MyDrive/M5_project"   # thư mục trên Google Drive để lưu dữ liệu và kết quả

from google.colab import drive
drive.mount("/content/drive")

import os
if not os.path.exists(f"/content/{REPO}"):
    !git clone https://github.com/{GITHUB_USER}/{REPO}.git /content/{REPO}
else:
    !git -C /content/{REPO} pull
%cd /content/{REPO}
!pip install -q -e .
!mkdir -p {DRIVE_DIR}/data_raw {DRIVE_DIR}/results

In [ ]:
# ===== DỮ LIỆU M5 =====
# Cách A (khuyến nghị): bạn đã có 3 tệp CSV trong Google Drive tại {DRIVE_DIR}/data_raw
#   calendar.csv, sell_prices.csv, sales_train_evaluation.csv
# Cách B: tải từ Kaggle (bỏ dấu # ở các dòng dưới, cần tệp kaggle.json của bạn)
# from google.colab import files
# files.upload()                       # chọn kaggle.json
# !mkdir -p ~/.kaggle && mv kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
# !bash scripts/download_m5.sh {DRIVE_DIR}/data_raw

!mkdir -p data
!ln -sfn {DRIVE_DIR}/data_raw data/raw
!ls -lh data/raw/

In [ ]:
RESULTS = f"{DRIVE_DIR}/results"       # mọi dự báo, bảng, nhật ký chạy lưu tại đây
import torch, lightgbm
print("GPU:", torch.cuda.is_available(), "| LightGBM", lightgbm.__version__, "| torch", torch.__version__)

## Bước 1 · Chạy thử trên dữ liệu GIẢ LẬP (không phải kết quả nghiên cứu)

In [ ]:
!python scripts/make_synthetic.py --out data/synthetic --items 4
!python experiments/run.py --exp B0 B1 L-HCP S-HCP --seeds 1 --raw-dir data/synthetic --n-series 30 --max-eval 2 --out results_smoke
!python experiments/evaluate.py --raw-dir data/synthetic --out results_smoke --pairs L-HCP:S-HCP

## Bước 2 · Đo thời gian trên 1 cửa sổ dữ liệu thật
Ghi lại thời gian của từng mô hình vào docs/NHAT_KY_THI_NGHIEM.md để ước lượng tổng thời gian.

In [ ]:
!python experiments/run.py --exp L-HCP S-HCP --seeds 1 --max-eval 1 --out {RESULTS}_timing

## Bước 3 · Baseline (không có tính ngẫu nhiên, chỉ chạy 1 lần)

In [ ]:
!python experiments/run.py --exp B0 B1 --out {RESULTS}

## Bước 4 · LightGBM: 3 bộ đặc trưng × 5 seed

In [ ]:
!python experiments/run.py --exp L-H L-HC L-HCP --seeds 1 2 3 4 5 --out {RESULTS}

## Bước 5 · LSTM: 3 bộ đặc trưng × 5 seed (cần GPU)
Nếu hết thời gian, chạy từng seed một: `--seeds 1`, rồi `--seeds 2`, …

In [ ]:
!python experiments/run.py --exp S-H S-HC S-HCP --seeds 1 2 3 4 5 --out {RESULTS}

## Bước 6 · Kiểm tra độ bền
R2 = LightGBM với lag ≥ 28 cho mọi tầm. R1 = DeepAR qua GluonTS (thử nghiệm, cần cài thêm).

In [ ]:
!python experiments/run.py --exp R2 --seeds 1 2 3 --out {RESULTS}
# !pip install -q "gluonts[torch]"
# !python experiments/run.py --exp R1 --seeds 1 2 3 --out {RESULTS}

## Bước 7 · Tinh chỉnh siêu tham số (tùy chọn, cùng ngân sách cho hai mô hình)
Xem hàm `tune_lgbm` trong src/m5cal/models/lgbm.py. Sau khi có tham số tốt nhất, cập nhật configs/lgbm.yaml, commit lên GitHub để ghi lại.

In [ ]:
import sys; sys.path.insert(0, "src")
import pandas as pd
from m5cal.config import load_config
from m5cal.pipeline import prepare
from m5cal.models.lgbm import tune_lgbm
from m5cal.metrics import attach_truth, series_window_scores, aggregate_scores

cfg = load_config("configs")
panel, calf, windows = prepare(cfg, out_dir=RESULTS)
tune_w = [w for w in windows if w.role == "tune"]
score = lambda preds: aggregate_scores(series_window_scores(panel, attach_truth(panel, preds), windows))["wrmsse"].mean()
# best_params, study = tune_lgbm(panel, calf, tune_w, cfg["lgbm"]["params"], cfg["lgbm"]["tuning"]["space"],
#                                "HCP", "weekly", n_trials=cfg["lgbm"]["tuning"]["n_trials"],
#                                history_days=cfg["base"]["data"]["history_days"], score_fn=score)
# best_params